In [ ]:
#导入模块
import numpy as np
from enum import Enum
import os
import cv2
import torch
import time
import torchvision
from easydict import EasyDict
import torch.nn as nn
import SimpleITK as sitk
import glob

In [ ]:
# Dicom2Nii,批次处理

dicom_root = r"\Dicom"       # 所有 DICOM 文件夹的根目录
nii_out_root = r"\NII"     # 输出 nii.gz 的目录

os.makedirs(nii_out_root, exist_ok=True)

for case_name in os.listdir(dicom_root):
    case_path = os.path.join(dicom_root, case_name)

    if not os.path.isdir(case_path):
        continue

    try:
        reader = sitk.ImageSeriesReader()
        dicom_names = reader.GetGDCMSeriesFileNames(case_path)

        if len(dicom_names) == 0:
            print(f"[跳过] {case_name}：未发现 DICOM")
            continue

        reader.SetFileNames(dicom_names)
        image = reader.Execute()

        out_path = os.path.join(nii_out_root, f"{case_name}.nii.gz")
        sitk.WriteImage(image, out_path)

        print(f"[完成] {case_name}")

    except Exception as e:
        print(f"[失败] {case_name}，原因：{e}")


In [ ]:
# Resample

def resample_nii_batch(input_folder, output_folder=None, target_spacing=(0.2, 0.2, 0.6)):
    """
    批量重采样NIfTI文件到统一体素间距
    参数:
        input_folder: 输入文件夹
        output_folder: 输出文件夹，如果为None则创建'resampled'子文件夹
        target_spacing: 目标体素间距(mm)
    """
    
    # 检查输出文件夹
    if output_folder is None:
        output_folder = os.path.join(input_folder, "resampled")
    
    os.makedirs(output_folder, exist_ok=True)
    
    # 查找所有NIfTI文件
    nii_files = glob.glob(os.path.join(input_folder, "*.nii*"))
    
    print(f"找到 {len(nii_files)} 个文件")
    print(f"目标体素间距: {target_spacing} mm")
    print("=" * 50)
    
    for i, nii_path in enumerate(nii_files):
        print(f"\n处理文件 {i+1}/{len(nii_files)}: {os.path.basename(nii_path)}")
        
        # 读取图像
        img = sitk.ReadImage(nii_path)
        
        # 获取原始信息
        original_spacing = img.GetSpacing()
        original_size = img.GetSize()
        
        print(f"  原始间距: {original_spacing} mm")
        print(f"  原始尺寸: {original_size}")
        
        # 计算新尺寸
        new_size = [
            int(round(original_size[0] * original_spacing[0] / target_spacing[0])),
            int(round(original_size[1] * original_spacing[1] / target_spacing[1])),
            int(round(original_size[2] * original_spacing[2] / target_spacing[2]))
        ]
        
        # 创建重采样器
        resampler = sitk.ResampleImageFilter()
        resampler.SetInterpolator(sitk.sitkLinear)
        resampler.SetOutputSpacing(target_spacing)
        resampler.SetSize(new_size)
        
        # 保持物理空间位置
        resampler.SetOutputOrigin(img.GetOrigin())
        resampler.SetOutputDirection(img.GetDirection())
        
        # 执行重采样
        resampled_img = resampler.Execute(img)
        
        # 保存文件
        filename = os.path.basename(nii_path)
        name, ext = os.path.splitext(filename)
        if ext == '.gz':
            name = os.path.splitext(name)[0]  # 处理.nii.gz
            
        output_path = os.path.join(output_folder, f"{name}_resampled.nii.gz")
        sitk.WriteImage(resampled_img, output_path)
        
        # 验证
        verify_img = sitk.ReadImage(output_path)
        print(f"  新尺寸: {verify_img.GetSize()}")
        print(f"  新间距: {verify_img.GetSpacing()} mm")
        print(f"  保存到: {output_path}")
    
    print("\n" + "=" * 50)
    print(f"批量重采样完成!")
    print(f"输出目录: {output_folder}")
    print(f"处理文件数: {len(nii_files)}")

# 使用示例
if __name__ == "__main__":
    input_dir = r"\NII"
    resample_nii_batch(input_dir)

In [2]:
#yolo部分代码
def preproc(img, input_size, swap=(2, 0, 1)):
    if len(img.shape) == 3:
        padded_img = np.ones((input_size[0], input_size[1], 3), dtype=np.uint8) * 114
    else:
        padded_img = np.ones(input_size, dtype=np.uint8) * 114

    r = min(input_size[0] / img.shape[0], input_size[1] / img.shape[1])
    resized_img = cv2.resize(
        img,
        (int(img.shape[1] * r), int(img.shape[0] * r)),
        interpolation=cv2.INTER_LINEAR,
    ).astype(np.uint8)
    padded_img[: int(img.shape[0] * r), : int(img.shape[1] * r)] = resized_img

    padded_img = padded_img.transpose(swap)
    padded_img = np.ascontiguousarray(padded_img, dtype=np.float32)
    return padded_img, r

def yolo_preprocess(img,input_size,swap=(2, 0, 1),legacy=False):
    img, _ = preproc(img, input_size, swap)
    if legacy:
        img = img[::-1, :, :].copy()
        img /= 255.0
        img -= np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
        img /= np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    return img, np.zeros((1, 5))

def yolo_postprocess(prediction, num_classes, conf_thre=0.7, nms_thre=0.45, class_agnostic=False):
    box_corner = prediction.new(prediction.shape)
    box_corner[:, :, 0] = prediction[:, :, 0] - prediction[:, :, 2] / 2
    box_corner[:, :, 1] = prediction[:, :, 1] - prediction[:, :, 3] / 2
    box_corner[:, :, 2] = prediction[:, :, 0] + prediction[:, :, 2] / 2
    box_corner[:, :, 3] = prediction[:, :, 1] + prediction[:, :, 3] / 2
    prediction[:, :, :4] = box_corner[:, :, :4]

    output = [None for _ in range(len(prediction))]
    for i, image_pred in enumerate(prediction):

        # If none are remaining => process next image
        if not image_pred.size(0):
            continue
        # Get score and class with highest confidence
        class_conf, class_pred = torch.max(image_pred[:, 5: 5 + num_classes], 1, keepdim=True)

        conf_mask = (image_pred[:, 4] * class_conf.squeeze() >= conf_thre).squeeze()
        # Detections ordered as (x1, y1, x2, y2, obj_conf, class_conf, class_pred)
        detections = torch.cat((image_pred[:, :5], class_conf, class_pred.float()), 1)
        detections = detections[conf_mask]
        if not detections.size(0):
            continue

        if class_agnostic:
            nms_out_index = torchvision.ops.nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                nms_thre,
            )
        else:
            nms_out_index = torchvision.ops.batched_nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                detections[:, 6],
                nms_thre,
            )

        detections = detections[nms_out_index]
        if output[i] is None:
            output[i] = detections
        else:
            output[i] = torch.cat((output[i], detections))

    return output


VOC_CLASSES = (
    "hsc", # 水平半规管
    "iac", # 内耳道
)

class Predictor(object):
    def __init__(
        self,
        model,
        attr,
        cls_names=VOC_CLASSES,
        trt_file=None,
        decoder=None,
        legacy=False,
    ):
        self.model = model
        self.cls_names = cls_names
        self.decoder = decoder
        self.num_classes = attr.num_classes
        self.confthre = attr.test_conf
        self.nmsthre = attr.nmsthre
        self.test_size = attr.test_size
        self.device = attr.device
        self.fp16 = attr.fp16
        self.legacy = legacy

    def inference(self, img):
        img_info = {"id": 0}
        if isinstance(img, str):
            img_info["file_name"] = os.path.basename(img)
            img = cv2.imread(img)
        else:
            img_info["file_name"] = None

        height, width = img.shape[:2]
        img_info["height"] = height
        img_info["width"] = width
        img_info["raw_img"] = img

        ratio = min(self.test_size[0] / img.shape[0], self.test_size[1] / img.shape[1])
        img_info["ratio"] = ratio

        img, _ = yolo_preprocess(img,self.test_size,legacy=self.legacy)
        img = torch.from_numpy(img).unsqueeze(0)
        img = img.float()
        if self.device == "gpu":
            img = img.cuda()
            if self.fp16:
                img = img.half()  # to FP16

        with torch.no_grad():
            t0 = time.time()
            outputs = self.model(img)
            if self.decoder is not None:
                outputs = self.decoder(outputs, dtype=outputs.type())
            outputs = yolo_postprocess(
                outputs, self.num_classes, self.confthre,
                self.nmsthre, class_agnostic=True
            )
            # logger.info("Infer time: {:.4f}s".format(time.time() - t0))
        return outputs, img_info
    
def get_model(attr):
    from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

    def init_yolo(M):
        for m in M.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eps = 1e-3
                m.momentum = 0.03

    in_channels = [256, 512, 1024]
    backbone = YOLOPAFPN(attr.depth, attr.width, in_channels=in_channels, act=attr.act)
    head = YOLOXHead(attr.num_classes, attr.width, in_channels=in_channels, act=attr.act)
    model = YOLOX(backbone, head)

    model.apply(init_yolo)
    model.head.initialize_biases(1e-2)
    model.train()
    return model

def get_yolo_model(ckpt_path):
    # attr
    attr = EasyDict()
    attr.num_classes = 2
    attr.test_conf = 0.25
    attr.nmsthre = 0.45
    attr.test_size = (640,640)
    attr.depth = 0.33
    attr.width = 0.50
    attr.act = 'silu' # 'relu'
    attr.fp16 = False
    attr.device = 'cpu'
    attr.save_result = True
    
    # init model
    model = get_model(attr)
    if torch.cuda.is_available():
        model.cuda()
        if attr.fp16:
            model.half()
        attr.device = 'gpu'
    model.eval()
    ckpt = torch.load(ckpt_path,map_location='cpu',weights_only=False)
    model.load_state_dict(ckpt['model'])
    return Predictor(model, attr, VOC_CLASSES)

In [3]:
#工具函数
class EAR_POS(Enum):
    LEFT=0,
    RIGHT=1,

class COMPUTE_MODE(Enum):
    UNKNOWN=0,
    JOINT=1,
    HSC=2,
    IAC=3,

def get_tb_region_new(hsc_bbox,iac_bbox,tb_pixelsize,ear_pos,compute_mode=1):
    preset_w = tb_pixelsize[0]
    preset_h = tb_pixelsize[1]

    # # tb_w, tb_h
    # if compute_mode == COMPUTE_MODE.JOINT:
    #     if ear_pos == EAR_POS.LEFT:
    #         tb_w = max(np.abs(hsc_bbox[2] - iac_bbox[0]) * 2, preset_w)
    #         tb_h = max((hsc_bbox[3] - iac_bbox[1] + (iac_bbox[3] - iac_bbox[1]))*2, preset_h)
    #     elif ear_pos == EAR_POS.RIGHT:
    #         tb_w = max((iac_bbox[2] - hsc_bbox[0])*2, preset_w)
    #         tb_h = max((hsc_bbox[3] - iac_bbox[1] + (iac_bbox[3] - iac_bbox[1]))*2,preset_h)
    # else:
    #     tb_w = preset_w
    #     tb_h = preset_h

    tb_w = preset_w
    tb_h = preset_h

    # tb_cx, tb_cy
    if compute_mode == COMPUTE_MODE.JOINT:
        if ear_pos == EAR_POS.LEFT:
            tb_cx = (iac_bbox[2] + hsc_bbox[0]) / 2 
        elif ear_pos == EAR_POS.RIGHT:
            tb_cx = (hsc_bbox[2] + iac_bbox[0]) / 2 
        tb_cy = hsc_bbox[1] 
    if compute_mode == COMPUTE_MODE.IAC: # center iac based
        if ear_pos == EAR_POS.LEFT:
            tb_cx = iac_bbox[2] # iac_xmax
        elif ear_pos == EAR_POS.RIGHT:
            tb_cx = iac_bbox[0] # iac_xmin
        tb_cy = iac_bbox[1] # iac_ymin
    else: # center hsc based
        if ear_pos == EAR_POS.LEFT:
            tb_cx = hsc_bbox[0] # hsc_xmin
        elif ear_pos == EAR_POS.RIGHT:
            tb_cx = hsc_bbox[2] # hsc_xmax
        tb_cy = hsc_bbox[1]

    tb_xmin = int(tb_cx - tb_w/2)
    tb_xmax = int(tb_cx + tb_w/2)
    tb_ymin = int(tb_cy - tb_h/2)
    tb_ymax = int(tb_cy + tb_h/2)
    
    tb_bbox = [tb_xmin,tb_ymin,tb_xmax,tb_ymax]
    return tb_bbox

def temporal_bone_detection(outputs,ear_pos,input_size,tb_pixelsize,topk=1):
    hsc_probs = outputs[:,4]*outputs[:,5] # 水平半规管
    iac_probs = outputs[:,11]*outputs[:,12] # 内听道
    joint_probs = hsc_probs*iac_probs # 联合概率
    # pdb.set_trace()
    compute_mode = COMPUTE_MODE.UNKNOWN
    # 若联合概率均为0，则使用hsc的概率和检测结果来确定tb区域
    if np.sum(joint_probs) != 0:
        det_probs = joint_probs
        compute_mode = COMPUTE_MODE.JOINT
    else:
        if np.sum(hsc_probs) != 0:
            det_probs = hsc_probs
            compute_mode = COMPUTE_MODE.HSC
        elif np.sum(iac_probs) != 0:
            det_probs = iac_probs
            compute_mode = COMPUTE_MODE.IAC

    print('\tear_pose: {} compute_mode: {}'.format(ear_pos,compute_mode))
    
    tb_bbox = []
    valid_topk_idxs = []
    if compute_mode != COMPUTE_MODE.UNKNOWN:
        argsort_idxs = (-det_probs).argsort()
        topk_idxs = argsort_idxs[:topk]
        topk_tb_bboxes = np.zeros((topk,4),dtype=np.float32)
        
        for i,idx in enumerate(topk_idxs):
            if det_probs[idx] == 0:
                continue
            hsc_bbox = outputs[idx,:4]
            iac_bbox = outputs[idx,7:11]
            # topk_tb_bboxes[i,:] = get_tb_region(hsc_bbox,iac_bbox,tb_pixelsize,ear_pos,compute_mode) # 颞骨区域提取
            topk_tb_bboxes[i,:] = get_tb_region_new(hsc_bbox,iac_bbox,tb_pixelsize,ear_pos,compute_mode)

            valid_topk_idxs.append(idx)

        if len(valid_topk_idxs) != 0:
            # 计算topk概率下的最大颞骨边界
            topk_tb_bboxes = topk_tb_bboxes[topk_tb_bboxes[:,2]!=0]
            tb_bbox_xmin = max(int(np.min(topk_tb_bboxes[:,0])),0)
            tb_bbox_ymin = max(int(np.min(topk_tb_bboxes[:,1])),0)
            tb_bbox_xmax = min(int(np.max(topk_tb_bboxes[:,2])),input_size[1])
            tb_bbox_ymax = min(int(np.max(topk_tb_bboxes[:,3])),input_size[0])
            
            tb_bbox_h = tb_bbox_ymax - tb_bbox_ymin
            tb_bbox_w = tb_bbox_xmax - tb_bbox_xmin
            
            tb_bbox_newh = max(tb_bbox_w,tb_bbox_h)
            tb_bbox_neww = tb_bbox_newh
            tb_bbox_nxmax = tb_bbox_xmin+tb_bbox_neww
            tb_bbox_nymax = tb_bbox_ymin+tb_bbox_newh
            
            tb_bbox = [tb_bbox_xmin,tb_bbox_ymin,tb_bbox_nxmax,tb_bbox_nymax]
    return tb_bbox,valid_topk_idxs

In [ ]:
import SimpleITK as sitk
def get_array_from_nii(nii_filepath,mask=0):
    data = sitk.ReadImage(nii_filepath)
    spacing = data.GetSpacing()
    origin = data.GetOrigin()

    img = sitk.GetArrayFromImage(data)
    if not mask:
        max = np.max(img)
        min = np.min(img)

        center = (max+min) / 2 
        window = max - min

        img = ((img - min) / window).astype(np.float32)
        img = (img * 255).astype(np.uint8)
        
    return img,spacing,origin
#核心前处理函数
def nii_process(nii_filepath, predictor, tb_physize, topk = 1):
    nii_array,spacing,origin = get_array_from_nii(nii_filepath)
    
    d,h,w = nii_array.shape
    nii_c_w = w/2
    input_size = (h,w)
    
    l_outputs = np.zeros((d,14),dtype=np.float32)
    r_outputs = np.zeros((d,14),dtype=np.float32)
    input = np.zeros((h,w,3),dtype=np.uint8)
    valid_slice_idxs = []
    for i in range(d):
        slice_img = nii_array[i,:,:]
        input[:,:,0] = slice_img
        input[:,:,1] = slice_img
        input[:,:,2] = slice_img
        # pdb.set_trace()
        outputs,img_info = predictor.inference(input)
        if outputs[0] == None:
            continue
        valid_slice_idxs.append(i)
        
        output = outputs[0].cpu().numpy()
        output[:,0:4] = output[:,0:4] / img_info['ratio']
        pred_num = output.shape[0]
        for j in range(pred_num):
            # pdb.set_trace()
            cur_pred = output[j,:]
            xmin = cur_pred[0]
            ymin = cur_pred[1]
            xmax = cur_pred[2]
            ymax = cur_pred[3]
            c_w = (xmin+xmax) / 2
            pred_label = int(cur_pred[-1])
            if c_w < nii_c_w:
                r_outputs[i,pred_label*7:pred_label*7+7] = cur_pred
            else:
                l_outputs[i,pred_label*7:pred_label*7+7] = cur_pred

        
        # # 左右同时出现检测框结果认为是有效
        # if np.sum(r_outputs[i,:]!=0) and np.sum(l_outputs[i,:]!=0):
        #     valid_slice_idxs.append(i)
    
    l_tb_bbox = []
    r_tb_bbox = []
    l_idxs = []
    r_idxs = []
    tb_bbox_zmin = 0
    tb_bbox_zmax = 0
    if len(valid_slice_idxs) >= 3:
    # compute pixel size
        tb_pixelsize = [0,0,0]
        tb_pixelsize[0] = tb_physize[0] / spacing[0] # w
        tb_pixelsize[1] = tb_physize[1] / spacing[1] # h
        tb_pixelsize[2] = tb_physize[2] / spacing[2] # d
        # print('\tvalid_slice_idxs: {}'.format(valid_slice_idxs))
        # # 取均值
        # valid_min_idx = np.min(valid_slice_idxs)
        # valid_max_idx = np.max(valid_slice_idxs)
        # tb_bbox_cz = (valid_min_idx+valid_max_idx)/2 
        # 取中位数
        valid_idx_num = len(valid_slice_idxs)
        if valid_idx_num % 2 == 0:
            median_idx = int(valid_idx_num/2)
            tb_bbox_cz = int((valid_slice_idxs[median_idx]+valid_slice_idxs[median_idx+1])/2)
        else:
            median_idx = int((valid_idx_num+1) / 2)
            tb_bbox_cz = valid_slice_idxs[median_idx]
        
        # crop_depth = min(max(valid_idx_num,tb_pixelsize[2]),d) # 裁切颞骨区域
        crop_depth = min(tb_pixelsize[2],d) # 内耳畸形裁切
        tb_bbox_zmin = max(0,int(tb_bbox_cz-crop_depth/2))
        tb_bbox_zmax = min(int(tb_bbox_cz+crop_depth/2),d)
        # pdb.set_trace()
        # slice temporal bone detction
        l_tb_bbox,l_idxs = temporal_bone_detection(l_outputs,EAR_POS.LEFT,input_size,tb_pixelsize,topk=topk)
        r_tb_bbox,r_idxs = temporal_bone_detection(r_outputs,EAR_POS.RIGHT,input_size,tb_pixelsize,topk=topk)
    
        
    return l_tb_bbox,l_idxs,r_tb_bbox,r_idxs,tb_bbox_zmin,tb_bbox_zmax

In [ ]:
#单文件调用示例

def savenii(vol0,spacing,origin,outname,std=False):
    if not std:
        vol = np.transpose(vol0, (2, 0, 1))
        vol = vol[::-1]
    else:
        vol=vol0
    out = sitk.GetImageFromArray(vol)
    out.SetSpacing(spacing)
    out.SetOrigin(origin)
    sitk.WriteImage(out,'%s.nii.gz'%(outname))



def test_single_case(nii_filepath,save_root,predictor,tb_physize,topk):
    print('process {}...'.format(nii_filepath))
    img,spacing,origin = get_array_from_nii(nii_filepath)
    nii_filename, _ = os.path.splitext(os.path.basename(nii_filepath))
    # savepath
    if not os.path.exists(save_root):
        os.mkdir(save_root)
    l_tb_filepath = os.path.join(save_root,'{}_left.nii.gz'.format(nii_filename))
    r_tb_filepath = os.path.join(save_root,'{}_right.nii.gz'.format(nii_filename))
    if os.path.exists(l_tb_filepath) or os.path.exists(r_tb_filepath):
        return
        
    # process
    l_tb_bbox,l_idxs,r_tb_bbox,r_idxs,zmin,zmax = nii_process(nii_filepath,predictor,tb_physize,topk=topk)
    if len(l_idxs) == 0 and  len(r_idxs) == 0:
        print('warning: abnormaly niifile!')
        return
    
    # crop and save
    # left
    if len(l_idxs) != 0:
        l_tb_img = img[zmin:zmax,l_tb_bbox[1]:l_tb_bbox[3],l_tb_bbox[0]:l_tb_bbox[2]]
        savenii(l_tb_img,spacing,origin,l_tb_filepath,std=True)
        #save_center_img(l_tb_img,tb_save_path,nii_filename,mode='left')
        # pdb.set_trace()
    else:
        print('\t{} left detect failed!'.format(nii_filename))
    # right
    if len(r_idxs) != 0:
        r_tb_img = img[zmin:zmax,r_tb_bbox[1]:r_tb_bbox[3],r_tb_bbox[0]:r_tb_bbox[2]]
        #save_center_img(r_tb_img,tb_save_path,nii_filename,mode='right')
        savenii(r_tb_img,spacing,origin,r_tb_filepath,std=True)
    else:
        print('\t{} right detect failed!'.format(nii_filename))


ckpt_file = './weights/best_ckpt.pth'
nii_filepaths = r"/data/zfti/szc/2026.3/0327trmp/in/base_in.nii.gz"
save_dir = r"/data/zfti/szc/2026.3/0327trmp/out3"
tb_psize = [20,20,20] # 预分割的颞骨物理尺寸(mm)
predictor = get_yolo_model(ckpt_file)
result = test_single_case(
    nii_filepath=nii_filepaths,
    predictor=predictor,
    tb_physize=tb_psize,
    save_root=save_dir,
    topk=5
) 